# 03 — Merge clinical and exposure data

This notebook:
- loads the clinical and exposure tables,
- keeps only the columns used later,
- cleans missing values,
- checks submitter IDs,
- creates `clinical_exposure_merged.tsv`.


In [47]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()

pd.set_option("display.max_columns", 120)



## 1. Define paths


In [48]:
clinical_path = PROJECT_ROOT / "data" / "clinical.tsv"
exposure_path = PROJECT_ROOT / "data" / "exposure.tsv"
merged_output_path = PROJECT_ROOT / "data" / "clinical_exposure_merged.tsv"

display(
    pd.DataFrame({
        "file": ["clinical", "exposure", "merged output"],
        "path": [clinical_path, exposure_path, merged_output_path],
    })
)



,file,path
0,clinical,/Users/michaljendrusak/PycharmProjects/tcga-lu...
1,exposure,/Users/michaljendrusak/PycharmProjects/tcga-lu...
2,merged output,/Users/michaljendrusak/PycharmProjects/tcga-lu...


## 2. Load the raw tables


In [49]:
NA_TOKENS = ["'--", "--", "Not Reported", "not reported", "Unknown", "unknown", ""]

clinical_raw = pd.read_csv(clinical_path, sep="	", na_values=NA_TOKENS, keep_default_na=True)
exposure_raw = pd.read_csv(exposure_path, sep="	", na_values=NA_TOKENS, keep_default_na=True)

display(
    pd.DataFrame({
        "table": ["clinical", "exposure"],
        "rows": [clinical_raw.shape[0], exposure_raw.shape[0]],
        "columns": [clinical_raw.shape[1], exposure_raw.shape[1]],
    })
)

display(clinical_raw.head())
display(exposure_raw.head())



,table,rows,columns
0,clinical,2466,210
1,exposure,522,40


,project.project_id,cases.case_id,cases.consent_type,cases.days_to_consent,cases.days_to_lost_to_followup,cases.disease_type,cases.index_date,cases.lost_to_followup,cases.primary_site,cases.submitter_id,demographic.age_at_index,demographic.age_is_obfuscated,demographic.cause_of_death,demographic.cause_of_death_source,demographic.country_of_birth,demographic.country_of_residence_at_enrollment,demographic.days_to_birth,demographic.days_to_death,demographic.demographic_id,demographic.education_level,demographic.ethnicity,demographic.gender,demographic.marital_status,demographic.occupation_duration_years,demographic.population_group,demographic.premature_at_birth,demographic.race,demographic.submitter_id,demographic.vital_status,demographic.weeks_gestation_at_birth,demographic.year_of_birth,demographic.year_of_death,diagnoses.adrenal_hormone,diagnoses.age_at_diagnosis,diagnoses.ajcc_clinical_m,diagnoses.ajcc_clinical_n,diagnoses.ajcc_clinical_stage,diagnoses.ajcc_clinical_t,diagnoses.ajcc_pathologic_m,diagnoses.ajcc_pathologic_n,diagnoses.ajcc_pathologic_stage,diagnoses.ajcc_pathologic_t,diagnoses.ajcc_serum_tumor_markers,diagnoses.ajcc_staging_system_edition,diagnoses.ann_arbor_b_symptoms,diagnoses.ann_arbor_b_symptoms_described,diagnoses.ann_arbor_clinical_stage,diagnoses.ann_arbor_extranodal_involvement,diagnoses.ann_arbor_pathologic_stage,diagnoses.best_overall_response,diagnoses.burkitt_lymphoma_clinical_variant,diagnoses.calgb_risk_group,diagnoses.cancer_detection_method,diagnoses.child_pugh_classification,diagnoses.clark_level,diagnoses.classification_of_tumor,diagnoses.cog_liver_stage,diagnoses.cog_neuroblastoma_risk_group,diagnoses.cog_renal_stage,diagnoses.cog_rhabdomyosarcoma_risk_group,...,diagnoses.uicc_clinical_stage,diagnoses.uicc_clinical_t,diagnoses.uicc_pathologic_m,diagnoses.uicc_pathologic_n,diagnoses.uicc_pathologic_stage,diagnoses.uicc_pathologic_t,diagnoses.uicc_staging_system_edition,diagnoses.ulceration_indicator,diagnoses.weiss_assessment_findings,diagnoses.weiss_assessment_score,diagnoses.who_cns_grade,diagnoses.who_nte_grade,diagnoses.wilms_tumor_histologic_subtype,diagnoses.year_of_diagnosis,treatments.chemo_concurrent_to_radiation,treatments.clinical_trial_indicator,treatments.course_number,treatments.days_to_treatment_end,treatments.days_to_treatment_start,treatments.drug_category,treatments.embolic_agent,treatments.initial_disease_status,treatments.lesions_treated_number,treatments.margin_distance,treatments.margin_status,treatments.margins_involved_site,treatments.number_of_cycles,treatments.number_of_fractions,treatments.prescribed_dose,treatments.prescribed_dose_units,treatments.pretreatment,treatments.protocol_identifier,treatments.radiosensitizing_agent,treatments.reason_treatment_ended,treatments.reason_treatment_not_given,treatments.regimen_or_line_of_therapy,treatments.residual_disease,treatments.route_of_administration,treatments.submitter_id,treatments.therapeutic_agents,treatments.therapeutic_level_achieved,treatments.therapeutic_levels_achieved,treatments.therapeutic_target_level,treatments.timepoint_category,treatments.treatment_anatomic_site,treatments.treatment_anatomic_sites,treatments.treatment_arm,treatments.treatment_dose,treatments.treatment_dose_max,treatments.treatment_dose_units,treatments.treatment_duration,treatments.treatment_effect,treatments.treatment_effect_indicator,treatments.treatment_frequency,treatments.treatment_id,treatments.treatment_intent_type,treatments.treatment_or_therapy,treatments.treatment_outcome,treatments.treatment_outcome_duration,treatments.treatment_type
0,TCGA-LUAD,0075437e-ba1a-46be-86d6-9773209a2b5e,Informed Consent,0.0,NaN,Adenomas and Adenocarcinomas,Diagnosis,No,Bronchus and lung,TCGA-62-A471,64.0,False,NaN,NaN,NaN,Germany,-23689.0,NaN,407caae1-7c07-52e1-bca7-1e60cac760eb,NaN,not hispanic or latino,male,NaN,NaN,NaN,NaN,white,TCGA-62-A471_demographic,Alive,NaN,NaN,NaN,NaN,23689.0,NaN,NaN,NaN,NaN,M0,N1,Stage IIB,T2b,NaN,7th,NaN,NaN,NaN,NaN,NaN,N

,project.project_id,cases.case_id,cases.submitter_id,exposures.age_at_last_exposure,exposures.age_at_onset,exposures.alcohol_days_per_week,exposures.alcohol_drinks_per_day,exposures.alcohol_frequency,exposures.alcohol_history,exposures.alcohol_intensity,exposures.alcohol_type,exposures.asbestos_exposure,exposures.asbestos_exposure_type,exposures.chemical_exposure_type,exposures.cigarettes_per_day,exposures.coal_dust_exposure,exposures.environmental_tobacco_smoke_exposure,exposures.exposure_duration,exposures.exposure_duration_hrs_per_day,exposures.exposure_duration_years,exposures.exposure_id,exposures.exposure_source,exposures.exposure_type,exposures.occupation_duration_years,exposures.occupation_type,exposures.pack_years_smoked,exposures.parent_with_radiation_exposure,exposures.radon_exposure,exposures.respirable_crystalline_silica_exposure,exposures.secondhand_smoke_as_child,exposures.smoking_frequency,exposures.submitter_id,exposures.time_between_waking_and_first_smoke,exposures.tobacco_smoking_onset_year,exposures.tobacco_smoking_quit_year,exposures.tobacco_smoking_status,exposures.type_of_smoke_exposure,exposures.type_of_tobacco_used,exposures.use_per_day,exposures.years_smoked
0,TCGA-LUAD,0075437e-ba1a-46be-86d6-9773209a2b5e,TCGA-62-A471,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tobacco,NaN,NaN,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006.0,Current Reformed Smoker for < or = 15 yrs,NaN,NaN,NaN,NaN
1,TCGA-LUAD,009be09b-f9f6-43b7-8f45-4a648f8123ce,TCGA-67-3773,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tobacco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Current Reformed Smoker for > 15 yrs,NaN,NaN,NaN,NaN
2,TCGA-LUAD,01e9888d-b5b9-48f1-8ba6-8a89af108a04,TCGA-NJ-A7XG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tobacco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Current Reformed Smoker for > 15 yrs,NaN,NaN,NaN,NaN
3,TCGA-LUAD,0232d299-4cdf-4fd7-9a5e-8d13c208b40c,TCGA-91-6848,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tobacco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007.0,Current Reformed Smoker for < or = 15 yrs,NaN,NaN,NaN,NaN
4,TCGA-LUAD,028e99e9-5b9a-4954-bb6e-6d4709a3cea8,TCGA-55-6986,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tobacco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lifelong Non-Smoker,NaN,NaN,NaN,NaN


## 3. Keep only the needed columns


In [50]:
clinical = clinical_raw[
    [
        "cases.submitter_id",
        "demographic.age_at_index",
        "demographic.gender",
        "demographic.race",
        "demographic.ethnicity",
    ]
].copy()

exposure = exposure_raw[
    [
        "cases.submitter_id",
        "exposures.tobacco_smoking_status",
        "exposures.pack_years_smoked",
    ]
].copy()

display(clinical.head())
display(exposure.head())

,cases.submitter_id,demographic.age_at_index,demographic.gender,demographic.race,demographic.ethnicity
0,TCGA-62-A471,64.0,male,white,not hispanic or latino
1,TCGA-62-A471,64.0,male,white,not hispanic or latino
2,TCGA-62-A471,64.0,male,white,not hispanic or latino
3,TCGA-62-A471,64.0,male,white,not hispanic or latino
4,TCGA-67-3773,84.0,female,white,not hispanic or latino


,cases.submitter_id,exposures.tobacco_smoking_status,exposures.pack_years_smoked
0,TCGA-62-A471,Current Reformed Smoker for < or = 15 yrs,30.0
1,TCGA-67-3773,Current Reformed Smoker for > 15 yrs,NaN
2,TCGA-NJ-A7XG,Current Reformed Smoker for > 15 yrs,NaN
3,TCGA-91-6848,Current Reformed Smoker for < or = 15 yrs,NaN
4,TCGA-55-6986,Lifelong Non-Smoker,NaN


## 4. Clean missing values


In [51]:
clinical_info_cols = [
    "demographic.age_at_index",
    "demographic.gender",
    "demographic.race",
    "demographic.ethnicity",
]

exposure_info_cols = [
    "exposures.tobacco_smoking_status",
    "exposures.pack_years_smoked",
]

clinical = clinical.dropna(subset=["cases.submitter_id"])
clinical = clinical[~clinical[clinical_info_cols].isna().all(axis=1)]
clinical = clinical.drop_duplicates(subset=["cases.submitter_id"], keep="first").copy()

exposure["removed_reason"] = pd.NA

exposure.loc[
    exposure["cases.submitter_id"].isna(),
    "removed_reason"
] = "missing patient ID"

exposure.loc[
    exposure["removed_reason"].isna()
    & exposure[exposure_info_cols].isna().all(axis=1),
    "removed_reason"
] = "all exposure fields missing"

exposure.loc[
    exposure["removed_reason"].isna()
    & exposure["cases.submitter_id"].duplicated(keep="first"),
    "removed_reason"
] = "duplicate patient ID"

removed_exposure_rows = exposure[exposure["removed_reason"].notna()].copy()
exposure = exposure[exposure["removed_reason"].isna()].drop(columns="removed_reason").copy()

display(
    pd.DataFrame({
        "step": [
            "before cleaning",
            "removed: missing patient ID",
            "removed: all exposure fields missing",
            "removed: duplicate patient ID",
            "after cleaning",
        ],
        "rows": [
            len(exposure) + len(removed_exposure_rows),
            (removed_exposure_rows["removed_reason"] == "missing patient ID").sum(),
            (removed_exposure_rows["removed_reason"] == "all exposure fields missing").sum(),
            (removed_exposure_rows["removed_reason"] == "duplicate patient ID").sum(),
            len(exposure),
        ],
    })
)

display(
    pd.DataFrame({
        "table": ["clinical", "exposure"],
        "rows_after_cleaning": [len(clinical), len(exposure)],
        "unique_ids": [
            clinical["cases.submitter_id"].nunique(),
            exposure["cases.submitter_id"].nunique(),
        ],
    })
)

display(removed_exposure_rows)

,step,rows
0,before cleaning,522
1,removed: missing patient ID,0
2,removed: all exposure fields missing,14
3,removed: duplicate patient ID,0
4,after cleaning,508


,table,rows_after_cleaning,unique_ids
0,clinical,522,522
1,exposure,508,508


,cases.submitter_id,exposures.tobacco_smoking_status,exposures.pack_years_smoked,removed_reason
30,TCGA-50-5935,NaN,NaN,all exposure fields missing
57,TCGA-50-5944,NaN,NaN,all exposure fields missing
111,TCGA-50-5930,NaN,NaN,all exposure fields missing
154,TCGA-50-8460,NaN,NaN,all exposure fields missing
186,TCGA-49-AAR3,NaN,NaN,all exposure fields missing
188,TCGA-50-5068,NaN,NaN,all exposure fields missing
251,TCGA-50-5933,NaN,NaN,all exposure fields missing
313,TCGA-50-5049,NaN,NaN,all exposure fields missing
331,TCGA-50-5045,NaN,NaN,all exposure fields missing
415,TCGA-50-5055,NaN,NaN,all exposure fields missing


## 5. Quick check of smoking-related columns


In [52]:
display(exposure["exposures.tobacco_smoking_status"].value_counts(dropna=False))

exposures.tobacco_smoking_status
Current Reformed Smoker for < or = 15 yrs          170
Current Reformed Smoker for > 15 yrs               137
Current Smoker                                     122
Lifelong Non-Smoker                                 75
Current Reformed Smoker, Duration Not Specified      4
Name: count, dtype: int64

## 6. Check missing values


In [53]:
clinical_missing = pd.DataFrame({
    "column": clinical.columns,
    "missing_n": clinical.isna().sum().values,
    "missing_pct": (clinical.isna().mean() * 100).round(2).values,
})
clinical_missing["table"] = "clinical"

exposure_missing = pd.DataFrame({
    "column": exposure.columns,
    "missing_n": exposure.isna().sum().values,
    "missing_pct": (exposure.isna().mean() * 100).round(2).values,
})
exposure_missing["table"] = "exposure"

missing_summary = pd.concat([clinical_missing, exposure_missing], ignore_index=True)
missing_summary = missing_summary[["table", "column", "missing_n", "missing_pct"]]
missing_summary = missing_summary.sort_values(["table", "missing_pct"], ascending=[True, False])

display(missing_summary)



,table,column,missing_n,missing_pct
4,clinical,demographic.ethnicity,126,24.14
3,clinical,demographic.race,67,12.84
1,clinical,demographic.age_at_index,19,3.64
0,clinical,cases.submitter_id,0,0.00
2,clinical,demographic.gender,0,0.00
7,exposure,exposures.pack_years_smoked,152,29.92
5,exposure,cases.submitter_id,0,0.00
6,exposure,exposures.tobacco_smoking_status,0,0.00


## 7. Check submitter IDs


In [54]:
id_check = pd.DataFrame({
    "table": ["clinical", "exposure"],
    "rows_after_cleaning": [len(clinical), len(exposure)],
    "unique_ids": [
        clinical["cases.submitter_id"].nunique(),
        exposure["cases.submitter_id"].nunique(),
    ],
    "all_ids_have_12_characters": [
        clinical["cases.submitter_id"].astype(str).str.len().eq(12).all(),
        exposure["cases.submitter_id"].astype(str).str.len().eq(12).all(),
    ],
    "duplicate_ids": [
        clinical["cases.submitter_id"].duplicated().sum(),
        exposure["cases.submitter_id"].duplicated().sum(),
    ],
})

display(id_check)



,table,rows_after_cleaning,unique_ids,all_ids_have_12_characters,duplicate_ids
0,clinical,522,522,True,0
1,exposure,508,508,True,0


## 8. Merge the tables


In [55]:
clinical_dedup = clinical.drop_duplicates(subset=["cases.submitter_id"], keep="first").copy()
exposure_dedup = exposure.drop_duplicates(subset=["cases.submitter_id"], keep="first").copy()

clin_dups_before = clinical["cases.submitter_id"].duplicated().sum()
expos_dups_before = exposure["cases.submitter_id"].duplicated().sum()

merged = pd.merge(
    exposure,
    clinical,
    on="cases.submitter_id",
    how="left",
    validate="one_to_one"
)

display(
    pd.DataFrame({
        "rows": [merged.shape[0]],
        "columns": [merged.shape[1]],
        "unique_patients": [merged["cases.submitter_id"].nunique()],
    })
)

display(merged.head())


,rows,columns,unique_patients
0,508,7,508


,cases.submitter_id,exposures.tobacco_smoking_status,exposures.pack_years_smoked,demographic.age_at_index,demographic.gender,demographic.race,demographic.ethnicity
0,TCGA-62-A471,Current Reformed Smoker for < or = 15 yrs,30.0,64.0,male,white,not hispanic or latino
1,TCGA-67-3773,Current Reformed Smoker for > 15 yrs,NaN,84.0,female,white,not hispanic or latino
2,TCGA-NJ-A7XG,Current Reformed Smoker for > 15 yrs,NaN,49.0,male,black or african american,not hispanic or latino
3,TCGA-91-6848,Current Reformed Smoker for < or = 15 yrs,NaN,59.0,male,white,not hispanic or latino
4,TCGA-55-6986,Lifelong Non-Smoker,NaN,74.0,female,white,NaN


## 9. Check missing values after merge


In [56]:
merged_missing = pd.DataFrame({
    "column": merged.columns,
    "missing_n": merged.isna().sum().values,
    "missing_pct": (merged.isna().mean() * 100).round(2).values,
}).sort_values("missing_pct", ascending=False)

display(merged_missing)

display(
    pd.DataFrame({
        "patients_with_smoking_annotation": [merged["exposures.tobacco_smoking_status"].notna().sum()]
    })
)



,column,missing_n,missing_pct
2,exposures.pack_years_smoked,152,29.92
6,demographic.ethnicity,125,24.61
5,demographic.race,67,13.19
3,demographic.age_at_index,19,3.74
0,cases.submitter_id,0,0.00
1,exposures.tobacco_smoking_status,0,0.00
4,demographic.gender,0,0.00


,patients_with_smoking_annotation
0,508


## 10. Save the merged table


In [57]:
merged.to_csv(merged_output_path, sep="	", index=False)

display(
    pd.DataFrame({
        "saved_file": [merged_output_path],
        "rows": [merged.shape[0]],
        "columns": [merged.shape[1]],
    })
)



,saved_file,rows,columns
0,/Users/michaljendrusak/PycharmProjects/tcga-lu...,508,7
